# Building Multimodal RAG Application #3: System Architecture

This companion notebook turns the article's eight-stage architecture into a small executable reference pipeline. It intentionally uses lightweight in-memory data so you can focus on how information moves through a multimodal RAG system before connecting production embedding, vector-database, and vision-language services.

## Architecture

1. Ingest enterprise images, video frames, audio transcripts, and text.
2. Preprocess and index the content in a vector database.
3. Accept a user query.
4. Encode the query with a multimodal embedding model.
5. Retrieve the most similar evidence.
6. Assemble the query and evidence into a prompt.
7. Send the prompt and visual evidence to a large vision-language model.
8. Return a grounded answer.

In [ ]:
from dataclasses import dataclass
from math import sqrt
from typing import Iterable

@dataclass(frozen=True)
class MultimodalRecord:
    record_id: str
    modality: str
    content: str
    embedding: tuple[float, ...]

## 1–2. Ingest, preprocess, and index enterprise data

In production, preprocessing can include frame sampling, OCR, speech-to-text, object detection, and normalization. Here the resulting descriptions and example embedding vectors stand in for those services.

In [ ]:
records = [
    MultimodalRecord("frame-001", "image", "A presenter points to a blue vector-database diagram.", (0.92, 0.81, 0.10)),
    MultimodalRecord("speech-001", "audio transcript", "The speaker explains similarity search over indexed embeddings.", (0.88, 0.76, 0.18)),
    MultimodalRecord("ocr-001", "on-screen text", "Vector database: retrieve the nearest multimodal records.", (0.95, 0.73, 0.12)),
    MultimodalRecord("frame-002", "image", "A red car travels along a mountain road.", (0.10, 0.15, 0.94)),
]

print(f"Indexed {len(records)} multimodal records.")

## 3–5. Encode a query and retrieve evidence

In [ ]:
def cosine_similarity(left: tuple[float, ...], right: tuple[float, ...]) -> float:
    dot_product = sum(a * b for a, b in zip(left, right))
    left_norm = sqrt(sum(value * value for value in left))
    right_norm = sqrt(sum(value * value for value in right))
    return dot_product / (left_norm * right_norm)

def retrieve(
    query_embedding: tuple[float, ...],
    index: Iterable[MultimodalRecord],
    top_k: int = 3,
) -> list[tuple[float, MultimodalRecord]]:
    ranked = [
        (cosine_similarity(query_embedding, record.embedding), record)
        for record in index
    ]
    return sorted(ranked, key=lambda item: item[0], reverse=True)[:top_k]

In [ ]:
query = "What does the video say about vector search?"
query_embedding = (0.91, 0.79, 0.14)  # Replace with a real multimodal encoder.
evidence = retrieve(query_embedding, records)

for score, record in evidence:
    print(f"{score:.3f} | {record.modality:16} | {record.content}")

## 6. Process the prompt

A production prompt processor can also attach the retrieved images or video frames themselves. This compact example serializes their descriptions and identifiers so the grounding contract remains visible.

In [ ]:
def build_prompt(query: str, evidence: list[tuple[float, MultimodalRecord]]) -> str:
    context = "\n".join(
        f"- [{record.record_id} | {record.modality}] {record.content}"
        for _, record in evidence
    )
    return (
        "Answer the question using only the retrieved multimodal evidence. "
        "Cite the record IDs that support the answer.\n\n"
        f"Question: {query}\n\nEvidence:\n{context}"
    )

prompt = build_prompt(query, evidence)
print(prompt)

## 7–8. LVLM inference and answer generation

The final boundary is deliberately represented as a function. Replace its body with your chosen LVLM client and pass any retrieved visual assets alongside the text prompt.

In [ ]:
def vision_language_model(prompt: str) -> str:
    # Deterministic stand-in for an LVLM call.
    assert "Evidence:" in prompt
    return (
        "The video explains that content is converted into embeddings and "
        "stored in a vector database, which retrieves the nearest records by "
        "similarity before generation [ocr-001, speech-001]."
    )

answer = vision_language_model(prompt)
print(answer)

## Production checklist

- Store source IDs, timestamps, modality, and access controls with every vector.
- Keep indexing and inference paths observable and independently retryable.
- Evaluate retrieval quality separately from answer quality.
- Pass original visual evidence to the LVLM when the answer depends on pixels.
- Preserve citations so users can inspect the evidence behind an answer.
- Add conversation memory only after defining its scope and retention policy.